# PyTorch 模型

In [ ]:
import sys
sys.path.extend([".."])
import set_env

In [ ]:
from tvm_book.tools.frontends import Frontend, TrainInputConfig
from tvm_book.tools import display

pytorch 前端模型配置：

In [ ]:
print(display.Tree("| ")("models/pytorch_demo"))

{icon}`fa fa-folder-open` `pytorch_demo/` 文件夹下存在如下内容：

- {icon}`fa fa-file` `save.py` 存储 PyTorch 模型为 `resnet18.pth`
    ```{include} models/pytorch_demo/save.py
    :code: python
    ```
- {icon}`fa fa-file` `resnet18.pth` 存储 PyTorch 模型结构与参数
- {icon}`fa fa-file` `config.toml` 存储 PyTorch 模型配置信息
    ```{include} models/pytorch_demo/config.toml
    :code: toml
    ```

In [ ]:
from pathlib import Path
import toml

config_path = "models/pytorch_demo/config.toml"

config_path = Path(config_path) 
config = toml.load(config_path)
model_type = config['model']["model_type"]
if len(config['train_inputs']) == 1:
    input_config = config['train_inputs'][0]
if model_type == "pytorch":
    shape_dict = {input_config["name"]: input_config["shape"]}
    model = Frontend(model_type).load(f"{config_path.parent}/{config['model']['path']}", shape_dict=shape_dict)

In [ ]:
import tvm
import tvm.relay as relay
from tvm.relay.build_module import bind_params_by_name
from tvm.ir.instrument import (
    PassTimingInstrument,
    pass_instrument,
)

In [81]:
@pass_instrument
class RelayGraphRecord:
    def __init__(self):
        self._op_diff = []
        # Passes can be nested.
        # Use stack to make sure we get correct before/after pairs.
        self._op_cnt_before_stack = []

    @staticmethod
    def _count_nodes(mod):
        """Count the number of occurrences of each operator in the module"""
        ret = {}

        def visit(node):
            if isinstance(node, relay.expr.Call):
                if hasattr(node.op, "name"):
                    op_name = node.op.name
                else:
                    # Some CallNode may not have 'name' such as relay.Function
                    return
                ret[op_name] = ret.get(op_name, 0) + 1

        relay.analysis.post_order_visit(mod["main"], visit)
        return ret

    # def enter_pass_ctx(self):
    #     self._op_diff = []
    #     self._op_cnt_before_stack = []

    # def exit_pass_ctx(self):
    #     assert len(self._op_cnt_before_stack) == 0, "The stack is not empty. Something wrong."

    # def run_before_pass(self, mod, info):
    #     self._op_cnt_before_stack.append((info.name, self._count_nodes(mod)))

    def run_after_pass(self, mod, info):
        # print(f"运行后 {info}")
        self._op_cnt_before_stack.append([info, mod])
    #     # Pop out the latest recorded pass.
    #     name_before, op_to_cnt_before = self._op_cnt_before_stack.pop()
    #     assert name_before == info.name, "name_before: {}, info.name: {} doesn't match".format(
    #         name_before, info.name
    #     )
    #     cur_depth = len(self._op_cnt_before_stack)
    #     op_to_cnt_after = self._count_nodes(mod)
    #     op_diff = self._diff(op_to_cnt_after, op_to_cnt_before)
    #     # only record passes causing differences.
    #     if op_diff:
    #         self._op_diff.append((cur_depth, info.name, op_diff))


In [82]:
import numpy as np
def get_calibration_dataset(mod, input_name):
    dataset = []
    input_shape = [int(x) for x in mod["main"].checked_type.arg_types[0].shape]
    for i in range(5):
        data = np.random.uniform(size=input_shape)
        dataset.append({input_name: data})
    return dataset

In [83]:
dataset = get_calibration_dataset(model.mod, "x")
record = RelayGraphRecord()
BASE_CFG = {
    "skip_conv_layers": [],
    "skip_dense_layers": False,
    "dtype_input": "int8",
    "dtype_weight": "int8",
    "dtype_activation": "int32",
}
with tvm.transform.PassContext(opt_level=3, instruments=[record]):
    with relay.quantize.qconfig(**BASE_CFG, calibrate_mode="percentile"):
        qmod = relay.quantize.quantize(model.mod, params=model.params, dataset=dataset)